LLM A Hands on appraoch project Gentut

In [1]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
os.makedirs('/content/drive/MyDrive/GenTut', exist_ok=True)
os.environ['HF_HOME'] = '/content/drive/MyDrive/GenTut/hf_cache'  # cache model weights across sessions

In [5]:
%cd /content/drive/MyDrive/GenTut
!git clone https://github.com/sandeshkg/gentut.git
%cd gentut

/content/drive/MyDrive/GenTut
Cloning into 'gentut'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 13 (delta 3), reused 10 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), done.
Resolving deltas: 100% (3/3), done.
/content/drive/MyDrive/GenTut/gentut


2. Install dependencies

In [6]:
%pip install -q langchain langgraph langchain-huggingface langchain-google-genai \
    transformers accelerate bitsandbytes pydantic streamlit google-generativeai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 37.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 94.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 70.5 MB/s eta 0:00:0000:0100:01


In [8]:
from dotenv import load_dotenv
import os

load_dotenv('/content/drive/MyDrive/GenTut/.env')

hf_token = os.getenv("HF_TOKEN")
gemini_key = os.getenv("GEMINI_API_KEY")

In [9]:
from huggingface_hub import login
login(token=hf_token)

#import google.generativeai as genai
#genai.configure(api_key=gemini_key)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [17]:
messages = [{"role": "user", "content": "Say hello in one sentence."}]
inputs = tokenizer.apply_chat_template(messages, 
                                    return_tensors="pt",
                                    add_generation_prompt=True,
                                    return_dict=True).to(model.device)

output = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<class 'transformers.tokenization_utils_base.BatchEncoding'>
KeysView({'input_ids': tensor([[128000, 128006,    882, 128007,    271,  46864,  24748,    304,    832,
          11914,     13, 128009, 128006,  78191, 128007,    271]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])})


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


user

Say hello in one sentence.assistant

Hello!


In [18]:
from schemas import CognitiveState
import json

schema_prompt = f"""You are a Skill Identifier agent. Given the student message below,
output ONLY a JSON object matching this schema (no prose, no markdown fences):
{CognitiveState.model_json_schema()}

Student message: "I don't understand why my for loop never terminates."
"""

messages = [{"role": "user", "content": schema_prompt}]
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True, return_dict=True).to(model.device)
output = model.generate(**inputs, max_new_tokens=300)
raw = tokenizer.decode(output[0][inputs.shape[-1]:], skip_special_tokens=True)

print(raw)
try:
    state = CognitiveState.model_validate_json(raw)
    print("✅ Valid:", state)
except Exception as e:
    print("❌ Failed validation:", e)

ModuleNotFoundError: No module named 'schemas'